In [3]:
import mysql.connector
conn= mysql.connector.connect(
    host="localhost",
    user="root",
    password="12345",
    database="bank_fraud",
    allow_local_infile=True
)
print("connected",conn.is_connected())

connected True


In [4]:
cursor = conn.cursor()

csv_path = r"C:\Users\dawas\Projects\Bank Fraud Detection\PS_20174392719_1491204439457_log.csv"

query = f"""
LOAD DATA LOCAL INFILE '{csv_path.replace("\\", "/")}'
INTO TABLE transactions
FIELDS TERMINATED BY ','
ENCLOSED BY '"'
LINES TERMINATED BY '\\n'
IGNORE 1 ROWS
"""

cursor.execute(query)
conn.commit()

print("Data imported successfully!")

Data imported successfully!


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn


In [3]:
df= pd.read_csv("PS_20174392719_1491204439457_log.csv")

In [6]:
df.head(10)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.0,0.00,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.0,0.00,0,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.0,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.0,0.00,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.0,0.00,0,0
5,1,PAYMENT,7817.71,C90045638,53860.00,46042.29,M573487274,0.0,0.00,0,0
6,1,PAYMENT,7107.77,C154988899,183195.00,176087.23,M408069119,0.0,0.00,0,0
7,1,PAYMENT,7861.64,C1912850431,176087.23,168225.59,M633326333,0.0,0.00,0,0
8,1,PAYMENT,4024.36,C1265012928,2671.00,0.00,M1176932104,0.0,0.00,0,0
9,1,DEBIT,5337.77,C712410124,41720.00,36382.23,C195600860,41898.0,40348.79,0,0


In [7]:
df.shape

(6362620, 11)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [9]:
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [10]:
df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00


In [11]:
pd.options.display.float_format='{:.3f}'.format

In [12]:
df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,6362620.000,6362620.000,6362620.000,6362620.000,6362620.000,6362620.000,6362620.000,6362620.000
mean,243.397,179861.904,833883.104,855113.669,1100701.667,1224996.398,0.001,0.000
std,142.332,603858.231,2888242.673,2924048.503,3399180.113,3674128.942,0.036,0.002
min,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,156.000,13389.570,0.000,0.000,0.000,0.000,0.000,0.000
50%,239.000,74871.940,14208.000,0.000,132705.665,214661.440,0.000,0.000
75%,335.000,208721.478,107315.175,144258.410,943036.708,1111909.250,0.000,0.000
max,743.000,92445516.640,59585040.370,49585040.370,356015889.350,356179278.920,1.000,1.000


In [13]:
df.duplicated().sum()

np.int64(0)

In [20]:
df['type'].unique()



array(['PAYMENT', 'TRANSFER', 'CASH_OUT', 'DEBIT', 'CASH_IN'],
      dtype=object)

In [15]:
df['step'].min(),df['step'].max()

(1, 743)

In [16]:
df['step'].nunique()

743

In [17]:
df['step'].describe().round(2)

count   6362620.000
mean        243.400
std         142.330
min           1.000
25%         156.000
50%         239.000
75%         335.000
max         743.000
Name: step, dtype: float64

In [18]:
df.groupby('step')['isFraud'].sum().describe()

count   743.000
mean     11.054
std       4.999
min       0.000
25%       8.000
50%      10.000
75%      14.000
max      40.000
Name: isFraud, dtype: float64

In [19]:
df.groupby('step')['isFraud'].sum().sort_values(ascending=False).head(10)

step
212    40
523    30
249    28
425    28
501    28
387    28
730    28
398    26
160    26
694    24
Name: isFraud, dtype: int64

In [21]:
step_analysis = df.groupby('step').agg(
    total_transactions=('isFraud', 'count'),
    fraud_transactions=('isFraud', 'sum')
)

step_analysis['fraud_rate'] = (
    step_analysis['fraud_transactions'] /
    step_analysis['total_transactions'] * 100
)

step_analysis.sort_values(
    'fraud_rate',
    ascending=False
).head(10)

,total_transactions,fraud_transactions,fraud_rate
step,,,
727,12,12,100.000
743,8,8,100.000
742,14,14,100.000
410,20,20,100.000
411,16,16,100.000
412,8,8,100.000
414,16,16,100.000
415,16,16,100.000
417,6,6,100.000


In [22]:
df['High_Value_Flag'] = (df['amount'] > 208721.48).astype(int)
df['High_Value_Flag'] 

0          0
1          0
2          0
3          0
4          0
          ..
6362615    1
6362616    1
6362617    1
6362618    1
6362619    1
Name: High_Value_Flag, Length: 6362620, dtype: int64

In [23]:
df['High_Value_Flag'].value_counts()

High_Value_Flag
0    4771965
1    1590655
Name: count, dtype: int64

In [24]:
pd.crosstab(df['High_Value_Flag'], df['isFraud'])

isFraud,0,1
High_Value_Flag,,
0,4769156,2809
1,1585251,5404


In [25]:
df['Risk_Score'] = df['High_Value_Flag']
df['Risk_Score'].value_counts()

Risk_Score
0    4771965
1    1590655
Name: count, dtype: int64

In [26]:
df['Expected_New_Balance'] = df['oldbalanceOrg'] - df['amount']

In [27]:
df['Balance_Anomaly_Flag'] = (
    abs(df['Expected_New_Balance'] - df['newbalanceOrig']) > 0.01
).astype(int)

In [28]:
df['Balance_Anomaly_Flag'].value_counts()

Balance_Anomaly_Flag
1    5077691
0    1284929
Name: count, dtype: int64

In [29]:
pd.crosstab(df['Balance_Anomaly_Flag'], df['isFraud'])

isFraud,0,1
Balance_Anomaly_Flag,,
0,1276761,8168
1,5077646,45


In [30]:
df['Expected_New_Dest_Balance'] = (
    df['oldbalanceDest'] + df['amount']
)
df['Dest_Balance_Anomaly_Flag'] = (
    abs(
        df['Expected_New_Dest_Balance'] - df['newbalanceDest']
    ) > 0.01
).astype(int)

In [31]:
df['Dest_Balance_Anomaly_Flag'].value_counts()
pd.crosstab(
    df['Dest_Balance_Anomaly_Flag'],
    df['isFraud']
)

isFraud,0,1
Dest_Balance_Anomaly_Flag,,
0,2170400,3573
1,4184007,4640


In [32]:
type_fraud = df.groupby('type').agg(
    Total_Transactions=('isFraud', 'count'),
    Fraud_Transactions=('isFraud', 'sum')
)

type_fraud['Fraud_Rate_%'] = (
    type_fraud['Fraud_Transactions']
    / type_fraud['Total_Transactions']
) * 100

type_fraud.sort_values('Fraud_Rate_%', ascending=False)

,Total_Transactions,Fraud_Transactions,Fraud_Rate_%
type,,,
TRANSFER,532909,4097,0.769
CASH_OUT,2237500,4116,0.184
CASH_IN,1399284,0,0.000
DEBIT,41432,0,0.000
PAYMENT,2151495,0,0.000


In [33]:
df['Type_Risk_Score'] = df['type'].map({
    'TRANSFER': 2,
    'CASH_OUT': 1,
    'CASH_IN': 0,
    'DEBIT': 0,
    'PAYMENT': 0
})

In [34]:
df['Type_Risk_Score'].value_counts()

Type_Risk_Score
0    3592211
1    2237500
2     532909
Name: count, dtype: int64

In [35]:
df['Risk_Score'] = (
    df['High_Value_Flag']
    + df['Type_Risk_Score']
)

In [36]:
df['Risk_Score'].value_counts().sort_index()

Risk_Score
0    3146415
1    1942358
2     869926
3     403921
Name: count, dtype: int64

In [37]:
risk_analysis = df.groupby('Risk_Score').agg(
    Total_Transactions=('isFraud', 'count'),
    Fraud_Transactions=('isFraud', 'sum')
)

risk_analysis['Fraud_Rate_%'] = (
    risk_analysis['Fraud_Transactions']
    / risk_analysis['Total_Transactions']
) * 100

risk_analysis

,Total_Transactions,Fraud_Transactions,Fraud_Rate_%
Risk_Score,,,
0,3146415,0,0.000
1,1942358,1418,0.073
2,869926,4089,0.470
3,403921,2706,0.670


In [38]:
risk_analysis.sort_index()

,Total_Transactions,Fraud_Transactions,Fraud_Rate_%
Risk_Score,,,
0,3146415,0,0.000
1,1942358,1418,0.073
2,869926,4089,0.470
3,403921,2706,0.670


In [39]:
def risk_category(score):
    if score == 0:
        return 'Low Risk'
    elif score == 1:
        return 'Medium Risk'
    else:
        return 'High Risk'

df['Risk_Category'] = df['Risk_Score'].apply(risk_category)

In [40]:
df['Risk_Category'].value_counts()

Risk_Category
Low Risk       3146415
Medium Risk    1942358
High Risk      1273847
Name: count, dtype: int64

In [41]:
pd.crosstab(
    df['Risk_Category'],
    df['isFraud'],
    normalize='index'
) * 100

isFraud,0,1
Risk_Category,,
High Risk,99.467,0.533
Low Risk,100.000,0.000
Medium Risk,99.927,0.073


## 🎯 KPI 1 — Total Transactions

In [42]:
total_transactions = df.shape[0]

print("Total Transactions:", total_transactions)

Total Transactions: 6362620


## 🎯 KPI 2 — Total Fraud Transactions

In [43]:
total_fraud = df['isFraud'].sum()

print("Total Fraud Transactions:", total_fraud)

Total Fraud Transactions: 8213


## 🎯 KPI 3 — Fraud Rate %


In [44]:
fraud_rate = (total_fraud / total_transactions) * 100

print("Fraud Rate: {:.2f}%".format(fraud_rate))

Fraud Rate: 0.13%


## KPI #4 — Total Fraud Amount

In [45]:
fraud_amount = df.loc[df['isFraud'] == 1, 'amount'].sum()

print("Total Fraud Amount: ₹{:,.2f}".format(fraud_amount))

Total Fraud Amount: ₹12,056,415,427.84


## 5 — Average Fraud Transaction Amount:

In [46]:
avg_fraud_amount = df.loc[df['isFraud'] == 1, 'amount'].mean()

print("Average Fraud Transaction Amount: ₹{:,.2f}".format(avg_fraud_amount))

Average Fraud Transaction Amount: ₹1,467,967.30


## 🎯 KPI #6 — High-Risk Transactions

In [47]:
high_risk_transactions = (df['Risk_Category'] == 'High Risk').sum()

print("High-Risk Transactions:", high_risk_transactions)

High-Risk Transactions: 1273847


In [48]:
high_risk_percentage = (
    high_risk_transactions / total_transactions
) * 100

print("High-Risk Transaction %: {:.2f}%".format(high_risk_percentage))

High-Risk Transaction %: 20.02%


In [49]:
high_risk_fraud = df.loc[
    df['Risk_Category'] == 'High Risk',
    'isFraud'
].sum()

print("Fraud Transactions in High Risk:", high_risk_fraud)

Fraud Transactions in High Risk: 6795


## 🎯 KPI — Fraud by Transaction Type

In [50]:
fraud_by_type = df.groupby('type').agg(
    Total_Transactions=('isFraud', 'count'),
    Fraud_Transactions=('isFraud', 'sum'),
    Fraud_Amount=('amount', lambda x: x[df.loc[x.index, 'isFraud'] == 1].sum())
)

fraud_by_type['Fraud_Rate_%'] = (
    fraud_by_type['Fraud_Transactions']
    / fraud_by_type['Total_Transactions']
) * 100

fraud_by_type.sort_values(
    'Fraud_Rate_%',
    ascending=False
)

,Total_Transactions,Fraud_Transactions,Fraud_Amount,Fraud_Rate_%
type,,,,
TRANSFER,532909,4097,6067213184.010,0.769
CASH_OUT,2237500,4116,5989202243.830,0.184
CASH_IN,1399284,0,0.000,0.000
DEBIT,41432,0,0.000,0.000
PAYMENT,2151495,0,0.000,0.000


## Fraud Trend by step

In [51]:
fraud_trend = df.groupby('step').agg(
    Total_Transactions=('isFraud', 'count'),
    Fraud_Transactions=('isFraud', 'sum'),
    Fraud_Amount=('amount', lambda x: x[df.loc[x.index, 'isFraud'] == 1].sum())
)

fraud_trend['Fraud_Rate_%'] = (
    fraud_trend['Fraud_Transactions']
    / fraud_trend['Total_Transactions']
) * 100

fraud_trend.head()

,Total_Transactions,Fraud_Transactions,Fraud_Amount,Fraud_Rate_%
step,,,,
1,2708,16,3740247.010,0.591
2,1014,8,4186592.480,0.789
3,552,4,66832.740,0.725
4,565,10,26400274.900,1.770
5,665,6,381841.540,0.902


In [52]:
fraud_trend.sort_values(
    'Fraud_Transactions',
    ascending=False
).head(10)

,Total_Transactions,Fraud_Transactions,Fraud_Amount,Fraud_Rate_%
step,,,,
212,34047,40,104335505.400,0.117
523,30,30,69540142.400,100.000
249,23209,28,47020241.040,0.121
425,28,28,110298471.230,100.000
501,28,28,79614672.480,100.000
387,28,28,147237648.390,100.000
730,28,28,115003435.050,100.000
398,26656,26,92473491.980,0.098
160,27765,26,88872442.840,0.094


In [53]:
high_volume_steps = fraud_trend[
    fraud_trend['Total_Transactions'] >= 10000
].sort_values(
    'Fraud_Transactions',
    ascending=False
)

high_volume_steps.head(10)

,Total_Transactions,Fraud_Transactions,Fraud_Amount,Fraud_Rate_%
step,,,,
212,34047,40,104335505.400,0.117
249,23209,28,47020241.040,0.121
160,27765,26,88872442.840,0.094
398,26656,26,92473491.980,0.098
406,12023,24,45330435.220,0.200
22,12635,23,11429114.240,0.182
34,30904,22,10074791.860,0.071
262,11125,22,23105143.940,0.198
250,31854,22,67870035.960,0.069


In [54]:
high_volume_steps.sort_values(
    'Fraud_Rate_%',
    ascending=False
).head(10)

,Total_Transactions,Fraud_Transactions,Fraud_Amount,Fraud_Rate_%
step,,,,
406,12023,24,45330435.220,0.200
262,11125,22,23105143.940,0.198
22,12635,23,11429114.240,0.182
261,14420,20,27005758.060,0.139
238,10920,14,31908350.780,0.128
249,23209,28,47020241.040,0.121
357,15096,18,37906720.700,0.119
212,34047,40,104335505.400,0.117
94,10372,12,14574168.340,0.116


In [55]:
risk_amount_analysis = df.groupby('Risk_Category').agg(
    Total_Transactions=('isFraud', 'count'),
    Fraud_Transactions=('isFraud', 'sum'),
    Total_Amount=('amount', 'sum'),
    Fraud_Amount=('amount', lambda x: x[df.loc[x.index, 'isFraud'] == 1].sum())
)

risk_amount_analysis

,Total_Transactions,Fraud_Transactions,Total_Amount,Fraud_Amount
Risk_Category,,,,
High Risk,1273847,6795,731402909122.170,11937686763.250
Low Risk,3146415,0,122227755163.180,0.000
Medium Risk,1942358,1418,290762280474.420,118728664.590


In [56]:
df.columns.tolist()

['step',
 'type',
 'amount',
 'nameOrig',
 'oldbalanceOrg',
 'newbalanceOrig',
 'nameDest',
 'oldbalanceDest',
 'newbalanceDest',
 'isFraud',
 'isFlaggedFraud',
 'High_Value_Flag',
 'Risk_Score',
 'Expected_New_Balance',
 'Balance_Anomaly_Flag',
 'Expected_New_Dest_Balance',
 'Dest_Balance_Anomaly_Flag',
 'Type_Risk_Score',
 'Risk_Category']

In [5]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 6362620
Columns: 11


In [57]:
df.drop(
    columns=[
        'Expected_New_Balance',
        'Expected_New_Dest_Balance'
    ],
    inplace=True
)

In [58]:
df.columns.tolist()

['step',
 'type',
 'amount',
 'nameOrig',
 'oldbalanceOrg',
 'newbalanceOrig',
 'nameDest',
 'oldbalanceDest',
 'newbalanceDest',
 'isFraud',
 'isFlaggedFraud',
 'High_Value_Flag',
 'Risk_Score',
 'Balance_Anomaly_Flag',
 'Dest_Balance_Anomaly_Flag',
 'Type_Risk_Score',
 'Risk_Category']

In [59]:
df[['Risk_Score', 'Risk_Category', 'High_Value_Flag',
    'Balance_Anomaly_Flag', 'Dest_Balance_Anomaly_Flag',
    'Type_Risk_Score']].head()

,Risk_Score,Risk_Category,High_Value_Flag,Balance_Anomaly_Flag,Dest_Balance_Anomaly_Flag,Type_Risk_Score
0,0,Low Risk,0,0,1,0
1,0,Low Risk,0,0,1,0
2,2,High Risk,0,0,1,2
3,1,Medium Risk,0,0,1,1
4,0,Low Risk,0,0,1,0


In [60]:
df.to_csv(
    'Bank_Fraud_Final.csv',
    index=False
)

In [61]:
import os

print("File created:", os.path.exists('Bank_Fraud_Final.csv'))

File created: True
